# Tutorial 02 — A single 2 mm raindrop at C-band

Falling raindrops are flattened by drag; larger drops are more
oblate. Dual-pol radar picks this up as differential reflectivity
Z_dr — the ratio of horizontally- to vertically-polarised
backscattered power. This notebook computes Z_dr, Z_h, and δ_hv
for a single drop and sweeps the axis ratio to show the
sensitivity.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from rustmatrix import Scatterer, radar
from rustmatrix.tmatrix_aux import (dsr_thurai_2007, geom_horiz_back,
                                      K_w_sqr, wl_C)
from rustmatrix.refractive import m_w_10C


## Build the drop and evaluate


In [ ]:
D_mm = 2.0
axis_ratio = 1.0 / dsr_thurai_2007(D_mm)
s = Scatterer(radius=D_mm/2, wavelength=wl_C, m=m_w_10C[wl_C],
              axis_ratio=axis_ratio, Kw_sqr=K_w_sqr[wl_C],
              ddelt=1e-4, ndgs=2)
s.set_geometry(geom_horiz_back)
print(f'axis ratio (h/v) = {axis_ratio:.4f}')
print(f'Z_h = {10*np.log10(radar.refl(s, True)):.2f} dBZ')
print(f'Z_dr = {10*np.log10(radar.Zdr(s)):+.3f} dB')
print(f'δ_hv = {np.degrees(radar.delta_hv(s)):+.4f}°')


## Z_dr across the 0.5–5 mm drop-size range


In [ ]:
Ds = np.linspace(0.5, 5.0, 20)
zdrs = np.empty_like(Ds)
for i, D in enumerate(Ds):
    s.radius = D / 2
    s.axis_ratio = 1.0 / dsr_thurai_2007(D)
    zdrs[i] = 10 * np.log10(radar.Zdr(s))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(Ds, zdrs, 'C0-o')
ax.set_xlabel('equivalent diameter D [mm]')
ax.set_ylabel('Z_dr [dB]')
ax.set_title('Single-drop Z_dr at C-band (Thurai 2007 shape)')
ax.grid(True, alpha=0.3)
fig.tight_layout();
